## Initiation

In [1]:
import pandas as pd
import numpy as np
import scipy.io as sio
import os
import logging


In [ ]:
# --- 1. 目录配置 (根据你最新定义的路径) ---
excluded_ids = [131]
RES_BASE_DIR = r"F:\NPI_result_2_7"
NET_EC_DIR = os.path.join(RES_BASE_DIR, "Net_EC_Results")
REPORT_PATH = os.path.join(RES_BASE_DIR, "Screening_Report.csv")
MED_RESULT_DIR = os.path.join(RES_BASE_DIR, "Mediation_Results")
os.makedirs(MED_RESULT_DIR, exist_ok=True)
med_log_path = os.path.join(RES_BASE_DIR, "med_log.log")
master_full_csv = os.path.join(RES_BASE_DIR, r"Master_Data_Full_177_with_EC.csv")

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(med_log_path),
        logging.StreamHandler()
    ]
)
logging.info("stat MEDIATION")

# network_names = ['Vis', 'SomMot', 'DorsAttn', 'SalVentAttn', 'Limbic', 'Cont', 'Default']
network_names = ['Vis', 'SMN', 'DAN', 'VAN', 'LMB', 'FPN', 'DMN']



2026-02-07 22:15:47,571 [INFO] stat MEDIATION


### * exclude

In [7]:

df = pd.read_csv(master_full_csv)

final_df = df[~df['Sub_ID'].isin(excluded_ids)].dropna().reset_index(drop=True)

# 4. 打印筛选报告，确保心里有数
logging.info(f"原始样本量: {len(df)}")
logging.info(f"手动排除 ID: {excluded_ids}")
logging.info(f"最终有效样本量: {len(final_df)}")

# 5. 保存用于统计分析的最终版本
final_save_path = os.path.join(RES_BASE_DIR, "Master_Data_after_excluding.csv")
final_df.to_csv(final_save_path, index=False)

logging.info(f"用于中介分析的干净数据已保存至: {final_save_path}")

2026-02-07 21:30:15,315 [INFO] 原始样本量: 177
2026-02-07 21:30:15,315 [INFO] 手动排除 ID: [131]
2026-02-07 21:30:15,316 [INFO] 最终有效样本量: 176
2026-02-07 21:30:15,319 [INFO] 用于中介分析的干净数据已保存至: F:\NPI_result_2_7\Master_Data_after_excluding.csv


## Mediation Analysis (Gemini version2.0)

In [3]:
import pingouin as pg
import pandas as pd
import os

# --- 1. 参数定义 ---
X = 'Group'
Ms = ['VAN_to_DMN_EC_Raw', 'VAN_to_DMN_EC_Exc', 'VAN_to_DMN_EC_Inh']
Ys = ['DMN_SP', 'VAN_to_DMN_Steps']
COVs = ['Age', 'Gender', 'MeanFD', 'Best_R'] # 加入 R 值作为技术协变量

# 确保数据目录存在
med_output_dir = os.path.join(RES_BASE_DIR, "Mediation_Results")
os.makedirs(med_output_dir, exist_ok=True)

# 加载数据
final_save_path = os.path.join(RES_BASE_DIR, "Master_Data_after_excluding.csv")
df_full = pd.read_csv(final_save_path)

# --- 2. 循环运行中介模型 ---
summary_results = []

for y_var in Ys:
    for m_var in Ms:
        logging.info("\n" + "="*60)
        logging.info(f"🚀 模型启动: {X} -> {m_var} -> {y_var}")
        
        # 准备数据并标准化
        current_cols = [X, m_var, y_var] + COVs
        df_model = df_full[current_cols].dropna()
        
        df_norm = df_model.copy()
        for col in [m_var, y_var] + COVs:
            df_norm[col] = (df_norm[col] - df_norm[col].mean()) / df_norm[col].std()

        # 运行中介分析
        res = pg.mediation_analysis(
            data=df_norm, x=X, m=m_var, y=y_var, 
            covar=COVs, alpha=0.05, n_boot=5000, seed=42
        )
        
        # --- 打印关键路径参数 ---
        # 我们从 res 中提取各路径的系数 (coef) 和 P值 (pval)
        def get_path_info(path_name):
            row = res[res['path'] == path_name]
            if not row.empty:
                return f"coef={row['coef'].values[0]:.4f}, p={row['pval'].values[0]:.4f}"
            return "N/A"

        logging.info(f"  [Path a]  (X -> M): {get_path_info(f'{m_var} ~ X')}")
        logging.info(f"  [Path b]  (M -> Y): {get_path_info(f'Y ~ {m_var}')}")
        logging.info(f"  [Direct]  (c'):     {get_path_info('Direct')}")
        logging.info(f"  [Total]   (c):      {get_path_info('Total')}")

        # --- 提取核心指标用于汇总 ---
        indirect_row = res[res['path'] == 'Indirect']
        indirect_coef = indirect_row['coef'].values[0]
        indirect_p = indirect_row['pval'].values[0]
        indirect_ci = [indirect_row['CI[2.5%]'].values[0], indirect_row['CI[97.5%]'].values[0]]
        
        total_coef = res.loc[res['path'] == 'Total', 'coef'].values[0]
        
        # 计算中介比例
        prop_med = indirect_coef / total_coef if total_coef != 0 else np.nan
        is_sig = (indirect_ci[0] * indirect_ci[1] > 0)
        
        logging.info(f"  [Indirect] (ab):   coef={indirect_coef:.4f}, 95% CI=[{indirect_ci[0]:.4f}, {indirect_ci[1]:.4f}]")
        logging.info(f"  [Proportion]:      {prop_med:.2%}")
        
        if is_sig:
            logging.info(f"✨ 发现显著中介效应! (CI不包含0)")
        else:
            logging.info(f"❌ 中介效应不显著。")

        # 存入汇总表
        summary_results.append({
            'Y': y_var,
            'M': m_var,
            'Path_a_p': res.loc[res['path'] == f'{m_var} ~ X', 'pval'].values[0],
            'Path_b_p': res.loc[res['path'] == f'Y ~ {m_var}', 'pval'].values[0],
            'Indirect_coef': indirect_coef,
            'Indirect_P': indirect_p,
            'CI_Lower': indirect_ci[0],
            'CI_Upper': indirect_ci[1],
            'Prop_Mediated': prop_med,
            'Significant': "YES" if is_sig else "NO"
        })
        
        # 保存详细 CSV
        res.to_csv(os.path.join(med_output_dir, f"Detail_{m_var}_to_{y_var}.csv"), index=False)

# --- 3. 生成最终汇总表 ---
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv(os.path.join(RES_BASE_DIR, "Mediation_Summary_Table.csv"), index=False)
logging.info("\n" + "="*60)
logging.info("所有中介模型分析完成。")

2026-02-07 22:15:52,009 [INFO] 
2026-02-07 22:15:52,010 [INFO] 🚀 模型启动: Group -> VAN_to_DMN_EC_Raw -> DMN_SP
2026-02-07 22:15:53,968 [INFO]   [Path a]  (X -> M): coef=-0.6207, p=0.0000
2026-02-07 22:15:53,970 [INFO]   [Path b]  (M -> Y): coef=0.0426, p=0.6012
2026-02-07 22:15:53,971 [INFO]   [Direct]  (c'):     coef=-0.5253, p=0.0008
2026-02-07 22:15:53,972 [INFO]   [Total]   (c):      coef=-0.4919, p=0.0008
2026-02-07 22:15:53,973 [INFO]   [Indirect] (ab):   coef=0.0334, 95% CI=[-0.0483, 0.1530]
2026-02-07 22:15:53,974 [INFO]   [Proportion]:      -6.79%
2026-02-07 22:15:53,975 [INFO] ❌ 中介效应不显著。
2026-02-07 22:15:53,978 [INFO] 
2026-02-07 22:15:53,978 [INFO] 🚀 模型启动: Group -> VAN_to_DMN_EC_Exc -> DMN_SP
2026-02-07 22:15:55,922 [INFO]   [Path a]  (X -> M): coef=-0.5282, p=0.0001
2026-02-07 22:15:55,924 [INFO]   [Path b]  (M -> Y): coef=0.0678, p=0.4164
2026-02-07 22:15:55,925 [INFO]   [Direct]  (c'):     coef=-0.4994, p=0.0012
2026-02-07 22:15:55,926 [INFO]   [Total]   (c):      coef=-0.49

## Mediation Analysis (Gemini version1.0)

In [ ]:
import pandas as pd
import pingouin as pg
import os

# --- 1. 加载数据 ---
data_path = os.path.join(med_result_dir, "Master_Data_after_excluding.csv")
df = pd.read_csv(data_path)

## 步骤2.1：计算中介变量M → VAN→DMN抑制性EC强度
df["VAN2DMN_Inh_Str"] = np.where(df["VAN_to_DMN_EC_Raw"] < 0, 
                                 abs(df["VAN_to_DMN_EC_Raw"]), 0)


## Shared variables

In [11]:
# ===================== 3. 定义通用变量（两个模型共用） =====================
# 核心变量映射（适配pingouin中介分析格式）
X = "Group"       # 自变量：分组（1=EOS，0=TD）
M_inh = "VAN2DMN_Inh_Str"  # 中介变量：抑制性EC强度
M = "VAN_to_DMN_EC_Raw"

COV = ["Gender", "Age", "MeanFD"]  # 控制变量：性别/年龄/头动
Y1 = "DMN_SP"     # 模型1因变量：DMN稳态概率
Y2 = "VAN_to_DMN_Steps"  # 模型2因变量：VAN→DMN最优步数

### 模型1因变量：DMN稳态概率

#### raw EC

In [12]:

# --- 3. 执行中介分析 ---
# n_boot=5000 是领域内公认的标准，用于非参数自助法检验
med_raw_sp = pg.mediation_analysis(
    data=df, 
    x=X, 
    m=M, 
    y=Y1, 
    covar=COV, 
    alpha=0.05, 
    n_boot=5000, 
    seed=42
)

# --- 4. 打印并保存结果 ---
print("--- 中介分析结果摘要 ---")
print(med_raw_sp)

med_raw_sp.to_csv(os.path.join(med_result_dir, f"Mediation_Result_{M}_to_{Y1}.csv"), index=False)

--- 中介分析结果摘要 ---
                    path      coef        se      pval  CI[2.5%]  CI[97.5%]  \
0  VAN_to_DMN_EC_Raw ~ X -0.000456  0.000101  0.000012 -0.000656  -0.000257   
1  Y ~ VAN_to_DMN_EC_Raw  1.047315  2.230133  0.639225 -3.354820   5.449451   
2                  Total -0.009895  0.003028  0.001310 -0.015872  -0.003918   
3                 Direct -0.010539  0.003209  0.001242 -0.016874  -0.004204   
4               Indirect  0.000644  0.001013  0.477200 -0.001017   0.003007   

   sig  
0  Yes  
1   No  
2  Yes  
3  Yes  
4   No  


#### EC inh

In [13]:

# --- 3. 执行中介分析 ---
# n_boot=5000 是领域内公认的标准，用于非参数自助法检验
med_inh_sp = pg.mediation_analysis(
    data=df, 
    x=X, 
    m=M_inh, 
    y=Y1, 
    covar=COV, 
    alpha=0.05, 
    n_boot=5000, 
    seed=42
)

# --- 4. 打印并保存结果 ---
print("--- 中介分析结果摘要 ---")
print(med_inh_sp)

med_inh_sp.to_csv(os.path.join(med_result_dir, f"Mediation_Result_{M}_to_{Y1}.csv"), index=False)

--- 中介分析结果摘要 ---
                  path      coef        se      pval   CI[2.5%]  CI[97.5%]  \
0  VAN2DMN_Inh_Str ~ X  0.000129  0.000033  0.000151   0.000063   0.000195   
1  Y ~ VAN2DMN_Inh_Str  2.505897  6.868851  0.715696 -11.052761  16.064556   
2                Total -0.009895  0.003028  0.001310  -0.015872  -0.003918   
3               Direct -0.011116  0.003150  0.000537  -0.017335  -0.004898   
4             Indirect  0.001221  0.000591  0.032000   0.000109   0.002460   

   sig  
0  Yes  
1   No  
2  Yes  
3  Yes  
4  Yes  


### 模型2：补充中介模型（Y=VAN_to_DMN_Steps，最优步数）

#### raw EC

In [14]:



# --- 3. 执行中介分析 ---
# n_boot=5000 是领域内公认的标准，用于非参数自助法检验
med_raw_step = pg.mediation_analysis(
    data=df, 
    x=X, 
    m=M, 
    y=Y2, 
    covar=COV, 
    alpha=0.05, 
    n_boot=5000, 
    seed=42
)

# --- 4. 打印并保存结果 ---
print("--- 中介分析结果摘要 ---")
print(med_raw_step)

med_raw_step.to_csv(os.path.join(med_result_dir, f"Mediation_Result_{M}_to_{Y2}.csv"), index=False)

--- 中介分析结果摘要 ---
                    path         coef          se          pval     CI[2.5%]  \
0  VAN_to_DMN_EC_Raw ~ X    -0.000456    0.000101  1.181577e-05    -0.000656   
1  Y ~ VAN_to_DMN_EC_Raw -1172.078172  154.664689  2.115349e-12 -1477.376052   
2                  Total     1.640441    0.216259  2.027743e-12     1.213559   
3                 Direct     1.237305    0.208970  1.724967e-08     0.824795   
4               Indirect     0.403136    0.104674  0.000000e+00     0.225352   

    CI[97.5%]  sig  
0   -0.000257  Yes  
1 -866.780292  Yes  
2    2.067323  Yes  
3    1.649815  Yes  
4    0.647308  Yes  


#### EC inh

In [15]:

# --- 3. 执行中介分析 ---
# n_boot=5000 是领域内公认的标准，用于非参数自助法检验
med_inh_step = pg.mediation_analysis(
    data=df, 
    x=X, 
    m=M_inh, 
    y=Y2, 
    covar=COV, 
    alpha=0.05, 
    n_boot=5000, 
    seed=42
)

# --- 4. 打印并保存结果 ---
print("--- 中介分析结果摘要 ---")
print(med_inh_step)

med_inh_step.to_csv(os.path.join(med_result_dir, f"Mediation_Result_{M}_to_{Y2}.csv"), index=False)

--- 中介分析结果摘要 ---
                  path         coef          se          pval    CI[2.5%]  \
0  VAN2DMN_Inh_Str ~ X     0.000129    0.000033  1.509196e-04    0.000063   
1  Y ~ VAN2DMN_Inh_Str  1838.688387  532.179580  6.939413e-04  788.201045   
2                Total     1.640441    0.216259  2.027743e-12    1.213559   
3               Direct     1.526382    0.224120  1.606077e-10    1.083965   
4             Indirect     0.114059    0.087552  2.088000e-01   -0.081204   

     CI[97.5%]  sig  
0     0.000195  Yes  
1  2889.175730  Yes  
2     2.067323  Yes  
3     1.968798  Yes  
4     0.261584   No  


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from mediation import mediate
from scipy.stats import zscore
import warnings
warnings.filterwarnings('ignore')  # 屏蔽无关警告，不影响结果

# ===================== 1. 配置参数与读取数据 =====================
# 替换为你的CSV文件实际路径（绝对路径/相对路径均可）

data_path = "Master_Data_after_excluding.csv"
# 读取数据，保留有效列（自动忽略无关列，适配你的数据）
df = pd.read_csv(data_path)[["Sub_ID", "Group", "DMN_SP", "VAN_to_DMN_Steps", 
                             "VAN_to_DMN_EC_Raw", "Gender", "Age", "MeanFD"]]

# ===================== 2. 核心预处理（2步：计算M+清洗数据） =====================
## 步骤2.1：从原始EC值计算【VAN→DMN抑制性EC强度】（中介变量M）
# 规则：原始EC<0→取绝对值（抑制性强度）；EC≥0→强度=0（无抑制）
df["VAN2DMN_Inh_Str"] = np.where(df["VAN_to_DMN_EC_Raw"] < 0, 
                                 abs(df["VAN_to_DMN_EC_Raw"]), 0)

## 步骤2.2：数据清洗（剔除缺失值+处理极端值，保证结果可靠）
# 剔除任意列缺失的被试
df = df.dropna()
# 对连续变量做Z-score极端值处理（|Z|>3视为极端值，替换为±3σ）
continuous_cols = ["DMN_SP", "VAN_to_DMN_Steps", "VAN2DMN_Inh_Str", "Age", "MeanFD"]
for col in continuous_cols:
    z = zscore(df[col])
    df[col] = np.where(z > 3, df[col].mean() + 3*df[col].std(),
                       np.where(z < -3, df[col].mean() - 3*df[col].std(), df[col]))
print(f"数据清洗完成，有效被试数：{len(df)}")

# ===================== 3. 定义通用变量（两个模型共用） =====================
# 自变量X：Group（1=EOS，0=TD）+ 控制变量（Gender/Age/MeanFD），加截距项
X = df[["Group", "Gender", "Age", "MeanFD"]]
X = sm.add_constant(X)  # 回归模型必须加截距，statsmodels默认不加
# 中介变量M：VAN→DMN抑制性EC强度（核心）
M = df["VAN2DMN_Inh_Str"]


### 模型1：核心中介模型（Y=DMN_SP，DMN稳态概率

In [ ]:

# ===================== 4. 模型1：核心中介模型（Y=DMN_SP，DMN稳态概率） =====================
print("\n" + "="*60)
print("【模型1：核心】X=Group → M=抑制性EC强度 → Y=DMN_SP（DMN稳态概率）")
print("="*60)
# 因变量Y1：DMN稳态概率
Y1 = df["DMN_SP"]
# 构建回归模型：X→M 与 X+M→Y1
model_m = sm.OLS(M, X).fit()  # X对M的回归（路径c：Group→M）
model_y1 = sm.OLS(Y1, sm.add_constant(pd.concat([X, M], axis=1))).fit()  # X+M对Y1的回归（路径b：M→Y1）
# Bootstrap中介分析（5000次抽样，95%CI，核心！）
med_result1 = mediate(
    model_m, model_y1,
    treat="Group",  # 核心自变量（EOS/TD分组）
    mediator="VAN2DMN_Inh_Str",  # 中介变量
    boot=True, n_rep=5000, conf_level=0.95,
    boot_type='percentile'  # 百分位法计算CI，适配神经影像数据
)
# 输出模型1核心结果
print(f"1. X→M路径（Group→抑制性EC强度）：系数={model_m.params['Group']:.6f}，p值={model_m.pvalues['Group']:.6f}")
print(f"2. M→Y路径（抑制性EC强度→DMN_SP）：系数={model_y1.params['VAN2DMN_Inh_Str']:.6f}，p值={model_y1.pvalues['VAN2DMN_Inh_Str']:.6f}")
print(f"3. 间接效应（X→M→Y）：估计值={med_result1.estimate:.6f}，95%CI=[{med_result1.conf_int[0]:.6f}, {med_result1.conf_int[1]:.6f}]")
print(f"4. 间接效应p值（Bootstrap）：{med_result1.p_value:.6f}")
print(f"5. 总效应：{med_result1.total_effect:.6f}，中介效应占比={(med_result1.estimate/med_result1.total_effect*100):.2f}%")
print(f"6. 中介效应显著性：{'显著（CI不含0）' if (med_result1.conf_int[0]>0 or med_result1.conf_int[1]<0) else '不显著（CI含0）'}")


In [ ]:

# ===================== 5. 模型2：补充中介模型（Y=VAN_to_DMN_Steps，最优步数） =====================
print("\n" + "="*60)
print("【模型2：补充】X=Group → M=抑制性EC强度 → Y=VAN_to_DMN_Steps（VAN→DMN最优步数）")
print("="*60)
# 因变量Y2：VAN→DMN最优步数
Y2 = df["VAN_to_DMN_Steps"]
# 构建回归模型：X→M（复用模型_m，无需重新拟合），X+M→Y2
model_y2 = sm.OLS(Y2, sm.add_constant(pd.concat([X, M], axis=1))).fit()
# Bootstrap中介分析（同模型1参数）
med_result2 = mediate(
    model_m, model_y2,
    treat="Group", mediator="VAN2DMN_Inh_Str",
    boot=True, n_rep=5000, conf_level=0.95, boot_type='percentile'
)
# 输出模型2核心结果
print(f"1. X→M路径（Group→抑制性EC强度）：系数={model_m.params['Group']:.6f}，p值={model_m.pvalues['Group']:.6f}（与模型1一致）")
print(f"2. M→Y路径（抑制性EC强度→最优步数）：系数={model_y2.params['VAN2DMN_Inh_Str']:.6f}，p值={model_y2.pvalues['VAN2DMN_Inh_Str']:.6f}")
print(f"3. 间接效应（X→M→Y）：估计值={med_result2.estimate:.6f}，95%CI=[{med_result2.conf_int[0]:.6f}, {med_result2.conf_int[1]:.6f}]")
print(f"4. 间接效应p值（Bootstrap）：{med_result2.p_value:.6f}")
print(f"5. 总效应：{med_result2.total_effect:.6f}，中介效应占比={(med_result2.estimate/med_result2.total_effect*100):.2f}%")
print(f"6. 中介效应显著性：{'显著（CI不含0）' if (med_result2.conf_int[0]>0 or med_result2.conf_int[1]<0) else '不显著（CI含0）'}")

# ===================== 6. 结果保存（可选，保存为CSV，方便论文整理） =====================
# 整理结果为DataFrame
result_df = pd.DataFrame({
    "模型": ["模型1（DMN_SP）", "模型2（最优步数）"],
    "X→M系数": [model_m.params['Group'], model_m.params['Group']],
    "X→M_p值": [model_m.pvalues['Group'], model_m.pvalues['Group']],
    "M→Y系数": [model_y1.params['VAN2DMN_Inh_Str'], model_y2.params['VAN2DMN_Inh_Str']],
    "M→Y_p值": [model_y1.pvalues['VAN2DMN_Inh_Str'], model_y2.pvalues['VAN2DMN_Inh_Str']],
    "间接效应估计值": [med_result1.estimate, med_result2.estimate],
    "95%CI下限": [med_result1.conf_int[0], med_result2.conf_int[0]],
    "95%CI上限": [med_result1.conf_int[1], med_result2.conf_int[1]],
    "间接效应p值": [med_result1.p_value, med_result2.p_value],
    "中介效应占比(%)": [(med_result1.estimate/med_result1.total_effect*100), (med_result2.estimate/med_result2.total_effect*100)],
    "显著性": ["显著" if (med_result1.conf_int[0]>0 or med_result1.conf_int[1]<0) else "不显著",
              "显著" if (med_result2.conf_int[0]>0 or med_result2.conf_int[1]<0) else "不显著"]
})
# 保存结果到CSV（同目录下，方便查看）
result_df.to_csv("中介分析结果汇总.csv", index=False, encoding="utf-8-sig")
print("\n✅ 所有结果已保存至：中介分析结果汇总.csv")